# AutoFeatureRec: автоматический поиск исторических признаков для рекомендаций KION

**Цель исследования**

Исследовать, насколько автоматическая генерация исторических признаков позволяет повысить качество рекомендательной модели по сравнению с ручным проектированием признаков.

**Основной исследовательский вопрос:** могут ли автоматически сгенерированные признаки обеспечить качество ранжирования, сопоставимое с ручным feature engineering, при приемлемых вычислительных затратах?

**Экспериментальный протокол**

* Данные: KION, история взаимодействий пользователей с контентом.

* Модель: CatBoost.

* Разбиение: временное разделение на train, validation и test.

* Признаки: ручные агрегаты, автоматически генерируемые временные статистики и поведенческие характеристики.

* Отбор признаков: по качеству на validation.

* Метрики: ROC-AUC, LogLoss, NDCG@10.

* Оценка устойчивости: bootstrap и сравнение результатов на разных выборках пользователей.

Для корректного сравнения моделей внутри каждого эксперимента используется одинаковый набор кандидатов.

**Ограничения исследования**

Датасет KION содержит агрегированные записи взаимодействий, поэтому last_watch_dt отражает дату последнего просмотра, а не полную последовательность пользовательских событий.

Отрицательные примеры формируются искусственно и не соответствуют реальным показам без взаимодействия. Поэтому результаты характеризуют качество ранжирования в рамках сформированного набора кандидатов, а не реальный CTR рекомендательной системы.

Дополнительно учитывается, что повторное использование результатов тестовых экспериментов при формировании новых гипотез ограничивает независимость итоговой оценки.

## 1. Зависимости и параметры

In [ ]:
!pip -q install catboost pyarrow

In [ ]:
from pathlib import Path
from datetime import timedelta
from collections import defaultdict
import os, time, random, zipfile, urllib.request, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score, log_loss
from IPython.display import display

SEED = 2026
N_USERS = 15000
NEG_PER_POS = 15
MAX_POS_PER_USER = 4
WINDOWS = [3, 7, 14, 30, 60, 120]
BASE_WINDOWS = [7, 30]
TRAIN_ITERATIONS = 200
OUT = Path('/content/autofeaturerec_final_2026'); OUT.mkdir(exist_ok=True)
DATA = Path('/content/kion_data'); DATA.mkdir(exist_ok=True)
random.seed(SEED); np.random.seed(SEED)
print('Параметры:', {'users':N_USERS, 'neg_per_pos':NEG_PER_POS, 'windows':WINDOWS})

Параметры: {'users': 15000, 'neg_per_pos': 15, 'windows': [3, 7, 14, 30, 60, 120]}


## 2. Получение KION и обзор схемы
Оригинальные файлы: [официальный репозиторий](https://github.com/irsafilo/KION_DATASET).

In [ ]:
archive = DATA / 'data_original.zip'
urls = [
    'https://github.com/irsafilo/KION_DATASET/raw/f69775be31fa5779907cf0a92ddedb70037fb5ae/data_original.zip',
    'https://raw.githubusercontent.com/irsafilo/KION_DATASET/main/data_original.zip',
    'https://github.com/irsafilo/KION_DATASET/raw/refs/heads/main/data_original.zip',
]
if not archive.exists() or not zipfile.is_zipfile(archive):
    if archive.exists(): archive.unlink()
    for url in urls:
        try:
            print('Загружаем:', url)
            urllib.request.urlretrieve(url, archive)
            if not zipfile.is_zipfile(archive): raise ValueError('Получен не ZIP')
            break
        except Exception as e:
            print('Попытка не удалась:', str(e)[:170])
            if archive.exists(): archive.unlink()
    else:
        raise RuntimeError('Не получилось загрузить ZIP. Откройте GitHub и загрузите data_original.zip вручную в /content/kion_data/')
with zipfile.ZipFile(archive) as z:
    print('В архиве:', z.namelist()[:12]); z.extractall(DATA)
files = {p.name.lower():p for p in DATA.rglob('*.csv')}
assert all(k in files for k in ('interactions.csv','users.csv','items.csv')), list(files)
interactions = pd.read_csv(files['interactions.csv'], parse_dates=['last_watch_dt'])
users = pd.read_csv(files['users.csv'])
items = pd.read_csv(files['items.csv'])
print('Размеры:', interactions.shape, users.shape, items.shape)
print('Колонки interactions:', interactions.columns.tolist())
display(interactions.head(3)); display(items.head(2))

Загружаем: https://github.com/irsafilo/KION_DATASET/raw/f69775be31fa5779907cf0a92ddedb70037fb5ae/data_original.zip
В архиве: ['data_original/', 'data_original/interactions.csv', '__MACOSX/data_original/._interactions.csv', 'data_original/users.csv', '__MACOSX/data_original/._users.csv', 'data_original/items.csv', '__MACOSX/data_original/._items.csv']
Размеры: (5476251, 5) (840197, 5) (15963, 14)
Колонки interactions: ['user_id', 'item_id', 'last_watch_dt', 'total_dur', 'watched_pct']


,user_id,item_id,last_watch_dt,total_dur,watched_pct
0,176549,9506,2021-05-11,4250,72.0
1,699317,1659,2021-05-29,8317,100.0
2,656683,7107,2021-05-09,10,0.0


,item_id,content_type,title,title_orig,release_year,genres,countries,for_kids,age_rating,studios,directors,actors,description,keywords
0,10711,film,Поговори с ней,Hable con ella,2002.0,"драмы, зарубежные, детективы, мелодрамы",Испания,NaN,16.0,NaN,Педро Альмодовар,"Адольфо Фернандес, Ана Фернандес, Дарио Гранди...",Мелодрама легендарного Педро Альмодовара «Пого...,"Поговори, ней, 2002, Испания, друзья, любовь, ..."
1,2508,film,Голые перцы,Search Party,2014.0,"зарубежные, приключения, комедии",США,NaN,16.0,NaN,Скот Армстронг,"Адам Палли, Брайан Хаски, Дж.Б. Смув, Джейсон ...",Уморительная современная комедия на популярную...,"Голые, перцы, 2014, США, друзья, свадьбы, прео..."


## 3. Сэмплирование пользователей: полная история каждого выбранного пользователя

In [ ]:
selected = np.random.default_rng(SEED).choice(interactions['user_id'].unique(), size=min(N_USERS, interactions['user_id'].nunique()), replace=False)
events = interactions.loc[interactions['user_id'].isin(selected), ['user_id','item_id','last_watch_dt','total_dur','watched_pct']].copy()
events = events.rename(columns={'last_watch_dt':'ts'}).dropna(subset=['ts'])
events['ts'] = pd.to_datetime(events['ts'])
events = events.sort_values(['ts','user_id','item_id']).reset_index(drop=True)
# Запись KION часто уже агрегирована на уровне пары user–item; не считаем несколько просмотров из неё.
events = events.drop_duplicates(['user_id','item_id','ts'])
print('События:',len(events),'пользователи:',events.user_id.nunique(),'объекты:',events.item_id.nunique())
print('Период:', events.ts.min(), '—', events.ts.max())
print('Средняя история:',round(len(events)/events.user_id.nunique(),2))
assert events.ts.notna().all()
del interactions

События: 84358 пользователи: 15000 объекты: 5798
Период: 2021-03-13 00:00:00 — 2021-08-22 00:00:00
Средняя история: 5.62


## 4. Категории фильмов и временные срезы
Жанр — статическая метаинформация, доступная до целевого взаимодействия. Train = первые 2 среза, validation = третий, test = четвёртый.

In [ ]:
assert 'item_id' in items.columns
meta = items[['item_id']].copy()
if 'genres' in items.columns:
    meta['genre'] = items['genres'].fillna('unknown').astype(str).str.split(',').str[0].str.strip().replace('', 'unknown')
else:
    meta['genre'] = 'unknown'
meta = meta.drop_duplicates('item_id')
# границы по календарному времени, а не по строкам
start, finish = events.ts.min(), events.ts.max()
fracs = [0.40,0.55,0.70,0.85,1.0]
bounds = [start + (finish-start)*f for f in fracs]
print('Срезы:', [(str(a.date()),str(b.date())) for a,b in zip(bounds[:-1],bounds[1:])])
assert len(set(bounds)) == 5, 'Очень короткий временной диапазон'
# хранение только нужных полей уменьшает память
events = events.merge(meta, on='item_id', how='left')
events['genre'] = events['genre'].fillna('unknown')
assert events['ts'].is_monotonic_increasing


Срезы: [('2021-05-16', '2021-06-10'), ('2021-06-10', '2021-07-04'), ('2021-07-04', '2021-07-28'), ('2021-07-28', '2021-08-22')]


## 5. Генерация кандидатов без заглядывания в будущее
На срезе каждый пользователь с хотя бы одной новой активностью получает положительные объекты в следующем окне и до `NEG_PER_POS` случайных кандидатов на каждый положительный. Каталог и частоты для выборки кандидатов определяются **только по прошлому**; не исключаем будущие просмотры за пределами оцениваемого окна.

In [ ]:
def make_candidates(history, future, snap_id, max_negs=NEG_PER_POS):
    rng = np.random.default_rng(SEED + snap_id)
    catalog = history['item_id'].drop_duplicates().to_numpy()
    if len(catalog) < 30:
        raise ValueError('Каталог в истории слишком маленький')
    # ограничиваем целевых пользователей теми, для кого есть история на момент среза
    known_users = set(history['user_id'].unique())
    active = future[future.user_id.isin(known_users)]
    seen = history.groupby('user_id')['item_id'].agg(set).to_dict()
    available = set(catalog)
    positives = (active[active.item_id.isin(available)]
                 .drop_duplicates(['user_id','item_id'])
                 .sort_values(['user_id','ts']))
    rows = []
    for uid, group in positives.groupby('user_id', sort=False):
        pos_ids = group['item_id'].to_numpy()[:MAX_POS_PER_USER]
        pos_set = set(group.item_id.tolist())  # предотвращаем false negatives для окна
        blocked = seen[uid] | pos_set
        rows.extend((snap_id, uid, int(i), 1) for i in pos_ids)
        eligible = np.array([i for i in catalog if i not in blocked], dtype=catalog.dtype)
        nneg = min(len(eligible), max_negs * len(pos_ids))
        if nneg:
            negs = rng.choice(eligible, size=nneg, replace=False)
            rows.extend((snap_id, uid, int(i), 0) for i in negs)
    result = pd.DataFrame(rows, columns=['snapshot','user_id','item_id','label'])
    result = result.merge(meta, on='item_id', how='left')
    result['genre'] = result['genre'].fillna('unknown')
    return result

snapshots = []
for snap_id, (cutoff, next_cutoff) in enumerate(zip(bounds[:-1],bounds[1:])):
    hist = events[events.ts < cutoff]
    fut = events[(events.ts >= cutoff) & (events.ts < next_cutoff if snap_id < 3 else events.ts <= next_cutoff)]
    cand = make_candidates(hist, fut, snap_id)
    assert not cand.duplicated(['snapshot','user_id','item_id']).any()
    assert cand.label.nunique() == 2, f'Нет обоих классов в срезе {snap_id}'
    snapshots.append(cand)
    print(f'{snap_id}: cutoff={cutoff.date()}, history={len(hist)}, future={len(fut)}, candidates={len(cand)}, positives={cand.label.sum()}, users={cand.user_id.nunique()}')


0: cutoff=2021-05-16, history=17326, future=10779, candidates=41408, positives=2588, users=1133
1: cutoff=2021-06-10, history=28105, future=16550, candidates=77824, positives=4864, users=1973
2: cutoff=2021-07-04, history=44655, future=17119, candidates=85776, positives=5361, users=2325
3: cutoff=2021-07-28, history=61774, future=22584, candidates=106896, positives=6681, users=2774


## Признаки: ручной baseline и автоматическое пространство поиска

Ручные признаки — исторические `count/nunique` по пользователю, фильму, жанру и паре пользователь–жанр (вся история, 7 и 30 дней). Автоматические признаки дополнительно включают окна 3/14/60/120 дней, статистики `sum/mean/std` по длительности и проценту просмотра, давность последнего события и долю жанра в истории пользователя. Все статистики строятся **строго по `ts < cutoff`**.

Важное ограничение: поскольку KION хранит агрегированную запись с датой последнего просмотра, поведенческие значения из такой записи нельзя гарантированно восстановить на более раннем срезе. Мы используем их лишь для записей с `last_watch_dt < cutoff` и явно фиксируем это допущение.

In [ ]:
KEYS = {'user':['user_id'], 'item':['item_id'], 'genre':['genre'], 'user_genre':['user_id','genre']}

def add_aggs(frame, hist, prefix, extended=False):
    out=frame
    for entity,keys in KEYS.items():
        gb=hist.groupby(keys,observed=True,dropna=False)
        counts=gb['item_id'].agg(['size','nunique']).reset_index().rename(columns={
            'size':f'{prefix}_{entity}_cnt','nunique':f'{prefix}_{entity}_nunique'})
        out=out.merge(counts,on=keys,how='left',validate='many_to_one')
        if extended:
            stats=gb.agg(dur_sum=('total_dur','sum'),dur_mean=('total_dur','mean'),
                         dur_std=('total_dur','std'),pct_mean=('watched_pct','mean'),
                         pct_std=('watched_pct','std'),last_ts=('ts','max')).reset_index()
            stats[f'{prefix}_{entity}_recency_days']=(CURRENT_CUTOFF-stats['last_ts']).dt.total_seconds()/86400
            stats=stats.drop(columns='last_ts').rename(columns={k:f'{prefix}_{entity}_{k}' for k in ['dur_sum','dur_mean','dur_std','pct_mean','pct_std']})
            out=out.merge(stats,on=keys,how='left',validate='many_to_one')
    if extended:
        num=f'{prefix}_user_genre_cnt'; den=f'{prefix}_user_cnt'
        out[f'{prefix}_user_genre_share']=out[num].fillna(0)/(out[den].fillna(0)+1e-6)
    return out

def create_features(candidates,cutoff,full_history):
    global CURRENT_CUTOFF
    CURRENT_CUTOFF=cutoff
    t0=time.monotonic()
    hist=full_history.loc[full_history.ts < cutoff]
    x=add_aggs(candidates.copy(),hist,'all',extended=True)
    for days in WINDOWS:
        h=hist.loc[hist.ts >= cutoff-pd.Timedelta(days=days)]
        x=add_aggs(x,h,f'w{days}',extended=True)
    numeric=[c for c in x.columns if c not in ['snapshot','user_id','item_id','label','genre']]
    x[numeric]=x[numeric].replace([np.inf,-np.inf],np.nan).fillna(0).astype('float32')
    x['genre']=x['genre'].astype(str)
    assert len(x)==len(candidates)
    assert (hist.ts<cutoff).all()
    return x,time.monotonic()-t0

assert {'total_dur','watched_pct'}.issubset(events.columns), 'Поведенческие поля потеряны при подготовке данных'
assert all(not c.duplicated(['snapshot','user_id','item_id']).any() for c in snapshots)
features=[]; costs=[]
for sid,(cand,cutoff) in enumerate(zip(snapshots,bounds[:-1])):
    x,seconds=create_features(cand,cutoff,events)
    features.append(x);costs.append(seconds)
    print(f'Snapshot {sid}: {len(x):,} pairs, {len(x.columns)} cols, {seconds:.2f}s')
all_data=pd.concat(features,ignore_index=True)
train=all_data.loc[all_data.snapshot.isin([0,1])].copy()
valid=all_data.loc[all_data.snapshot==2].copy()
test=all_data.loc[all_data.snapshot==3].copy()
print('Train / validation / test:',len(train),len(valid),len(test))
assert train.label.nunique()==valid.label.nunique()==test.label.nunique()==2


Snapshot 0: 41,408 pairs, 236 cols, 3.72s
Snapshot 1: 77,824 pairs, 236 cols, 6.61s
Snapshot 2: 85,776 pairs, 236 cols, 7.16s
Snapshot 3: 106,896 pairs, 236 cols, 9.19s
Train / validation / test: 119232 85776 106896


## Модели и поиск признаков на validation

Чтобы отделить полезность новых статистик от простого увеличения размерности, сравниваем: популярность, ручной baseline, расширенные ручные признаки и автоматические признаки. Ранжируем автоматически созданные признаки по importance модели, обученной **только на train**, а размер набора выбираем по validation. Тест до финального выбора не используется.

In [ ]:
from sklearn.metrics import roc_auc_score,log_loss

def ndcg_per_user(frame,pred,k=10):
    x=frame[['snapshot','user_id','label']].copy()
    x['score']=pred
    out=[]
    for (sid,uid),part in x.groupby(['snapshot','user_id'],sort=False):
        labels=part.sort_values('score',ascending=False).label.to_numpy()[:k]
        dcg=(labels/np.log2(np.arange(2,len(labels)+2))).sum()
        n=min(int(part.label.sum()),k)
        ideal=(1/np.log2(np.arange(2,n+2))).sum() if n else 0
        if ideal:out.append((sid,uid,dcg/ideal))
    return pd.DataFrame(out,columns=['snapshot','user_id','ndcg'])

def score(frame,pred):
    return {'ROC-AUC':roc_auc_score(frame.label,pred),
            'LogLoss':log_loss(frame.label,np.clip(pred,1e-7,1-1e-7)),
            'NDCG@10':ndcg_per_user(frame,pred).ndcg.mean()}

def fit(cols):
    m=CatBoostClassifier(iterations=200,depth=6,learning_rate=.07,l2_leaf_reg=5,
                         loss_function='Logloss',random_seed=SEED,verbose=False,
                         thread_count=2,allow_writing_files=False)
    t=time.monotonic()
    m.fit(train[cols],train.label,cat_features=[c for c in cols if train[c].dtype=='object'])
    return m,time.monotonic()-t

count_cols=[c for c in all_data if c.endswith(('_cnt','_nunique'))]
manual=['genre']+[c for c in count_cols if c.startswith('all_') or
                         (c.startswith(('w7_','w30_')) and c.endswith('_cnt'))]
manual=list(dict.fromkeys(manual))
all_features=['genre']+[c for c in all_data.columns if c not in
                          ['snapshot','user_id','item_id','label','genre']]
behavior_cols=[c for c in all_features if any(s in c for s in ['dur_','pct_','recency_days','_share'])]
manual_plus=list(dict.fromkeys(manual+[c for c in behavior_cols if c.startswith(('all_','w7_','w30_'))]))
assert len(behavior_cols)>0

rows=[];models={};sets={}
def evaluate(name,cols):
    m,seconds=fit(cols)
    v=score(valid,m.predict_proba(valid[cols])[:,1])
    rows.append({'model':name,'features':len(cols),'fit_s':seconds,**v})
    models[name]=m;sets[name]=cols
    print(name,len(cols),round(v['NDCG@10'],5),'fit_s',round(seconds,1))

evaluate('Popularity',['all_item_cnt'])
evaluate('Manual',manual)
evaluate('Manual_plus_behavior',manual_plus)
evaluate('Auto_all',all_features)
imp=pd.Series(models['Auto_all'].get_feature_importance(),index=all_features).drop('genre',errors='ignore').sort_values(ascending=False)
for k in [8,16,32,64,96]:
    evaluate(f'Auto_top{k}',['genre']+imp.head(k).index.tolist())
validation=pd.DataFrame(rows).sort_values('NDCG@10',ascending=False)
display(validation)
validation.to_csv(OUT/'validation_metrics.csv',index=False)
imp.to_csv(OUT/'feature_importance.csv',header=['importance'])
print('Генерация признаков, секунды:',costs)


Popularity 1 0.6919 fit_s 4.5
Manual 17 0.72455 fit_s 22.1
Manual_plus_behavior 92 0.72349 fit_s 32.2
Auto_all 232 0.72474 fit_s 49.7
Auto_top8 9 0.71021 fit_s 21.2
Auto_top16 17 0.71224 fit_s 22.7
Auto_top32 33 0.72142 fit_s 24.7
Auto_top64 65 0.72359 fit_s 28.7
Auto_top96 97 0.72409 fit_s 31.9


,model,features,fit_s,ROC-AUC,LogLoss,NDCG@10
3,Auto_all,232,49.717998,0.875738,0.151869,0.724742
1,Manual,17,22.141666,0.873595,0.151990,0.724552
8,Auto_top96,97,31.923575,0.874443,0.152654,0.724093
7,Auto_top64,65,28.698710,0.875487,0.152784,0.723589
2,Manual_plus_behavior,92,32.211809,0.874426,0.152542,0.723487
6,Auto_top32,33,24.742303,0.875613,0.151903,0.721420
5,Auto_top16,17,22.666385,0.865866,0.155098,0.712241
4,Auto_top8,9,21.196677,0.864433,0.155452,0.710209
0,Popularity,1,4.494946,0.846543,0.165625,0.691903


Генерация признаков, секунды: [3.7192970819999687, 6.614837588, 7.1579605130000346, 9.193607113000041]


## Итоговая оценка на test

Выбираем лучший **автоматический** набор только по validation; сравниваем с ручным baseline и вариантом с поведенческими признаками. Bootstrap ресэмплирует пользователей и оценивает неопределённость разницы средних NDCG@10.

In [ ]:
best_auto=validation.loc[validation.model.str.startswith('Auto_')].iloc[0]['model']
final_names=['Popularity','Manual','Manual_plus_behavior',best_auto]
final_rows=[];per_user={}
for name in dict.fromkeys(final_names):
    pred=models[name].predict_proba(test[sets[name]])[:,1]
    final_rows.append({'model':name,'features':len(sets[name]),**score(test,pred)})
    per_user[name]=ndcg_per_user(test,pred).set_index(['snapshot','user_id']).ndcg
final=pd.DataFrame(final_rows)
display(final)
final.to_csv(OUT/'test_metrics.csv',index=False)

# Парный bootstrap по пользователям
rng=np.random.default_rng(2026)
a=per_user[best_auto]; b=per_user['Manual']; common=a.index.intersection(b.index)
delta=(a.loc[common]-b.loc[common]).to_numpy()
boot=np.array([rng.choice(delta,size=len(delta),replace=True).mean() for _ in range(1500)])
ci=np.quantile(boot,[.025,.975])
print('Auto minus Manual:',round(delta.mean(),6),'95% CI:',np.round(ci,6),'n_users:',len(delta))
pd.DataFrame([{'best_auto':best_auto,'delta_ndcg':delta.mean(),
               'ci_low':ci[0],'ci_high':ci[1],'n_users':len(delta)}]).to_csv(OUT/'bootstrap.csv',index=False)


,model,features,ROC-AUC,LogLoss,NDCG@10
0,Popularity,1,0.857447,0.171243,0.665320
1,Manual,17,0.880403,0.153275,0.712939
2,Manual_plus_behavior,92,0.881008,0.154386,0.714604
3,Auto_all,232,0.882070,0.153878,0.714377


Auto minus Manual: 0.001438 95% CI: [-0.002407  0.005584] n_users: 2774


## Диагностика и воспроизводимость

Проверяем важность семейств признаков и сохраняем параметры. Это описательный анализ обученной модели, а не новый цикл подбора по test.

In [ ]:
def family(c):
    if 'recency_days' in c:return 'recency'
    if '_share' in c:return 'share'
    if any(x in c for x in ['dur_','pct_']):return 'behavior'
    if c.endswith(('_cnt','_nunique')):return 'counts'
    return 'categorical'
family_table=(imp.rename('importance').reset_index().rename(columns={'index':'feature'})
              .assign(family=lambda d:d.feature.map(family))
              .groupby('family',as_index=False).agg(n_features=('feature','size'),total_importance=('importance','sum')))
display(family_table)
family_table.to_csv(OUT/'importance_by_family.csv',index=False)
config={'seed':SEED,'users':N_USERS,'negatives_per_positive':NEG_PER_POS,
        'windows':WINDOWS,'selected_auto':best_auto,'selected_features':sets[best_auto],
        'catboost_iterations':200,'catboost_depth':6,'catboost_learning_rate':.07,
        'feature_generation_seconds':costs,
        'limitations':['last_watch_dt is aggregated','sampled unexposed negatives','future-active user conditioning']}
(OUT/'config.json').write_text(json.dumps(config,ensure_ascii=False,indent=2),encoding='utf-8')
print('Артефакты:',*[p.name for p in OUT.iterdir()],sep='\n - ')


,family,n_features,total_importance
0,behavior,140,60.005258
1,counts,56,23.223308
2,recency,28,14.022045
3,share,7,2.749390


Артефакты:
 - config.json
 - validation_metrics.csv
 - importance_by_family.csv
 - feature_importance.csv
 - bootstrap.csv
 - test_metrics.csv
